In [1]:
import polars as pl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report,
)
from sentence_transformers import SentenceTransformer

/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 1. Define your local file paths
train_path = "/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/train-00000-of-00001.parquet"
test_path = "/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/test-00000-of-00001.parquet"

# 2. Load the data (use pl.read_csv if they are CSV files)
df_train = pl.read_parquet(train_path)
df_test = pl.read_parquet(test_path)

# 3. Quick inspect
print(df_train.head(3))

shape: (3, 2)
┌─────────────────────────────────┬───────┐
│ text                            ┆ label │
│ ---                             ┆ ---   │
│ str                             ┆ i64   │
╞═════════════════════════════════╪═══════╡
│ I am still waiting on my card?  ┆ 11    │
│ What can I do if my card still… ┆ 11    │
│ I have been waiting over a wee… ┆ 11    │
└─────────────────────────────────┴───────┘


- Data is imbalanced
- Test data is balanced so its not representinf training distirbution (is this good or bad)
- there are 77 classes

In [6]:
# --- Number of classes ---
LABEL_COL = "label"
n_classes = df_train[LABEL_COL].n_unique()
print(f"Number of classes: {n_classes}\n")

# --- Support per class in TRAIN (sorted ascending, so rarest are at the top) ---
print("=== TRAIN support per class ===")
train_counts = (
    df_train[LABEL_COL]
    .value_counts()
    .sort("count")                    # ascending: smallest classes first
)
print(train_counts)

# --- Support per class in TEST ---
print("\n=== TEST support per class ===")
test_counts = (
    df_test[LABEL_COL]
    .value_counts()
    .sort("count")
)
print(test_counts)

# --- Quick imbalance summary for TRAIN ---
counts = train_counts["count"]
print("\n=== TRAIN imbalance summary ===")
print(f"Total examples : {len(df_train)}")
print(f"Min support    : {counts.min()}")
print(f"Max support    : {counts.max()}")
print(f"Mean support   : {counts.mean():.1f}")
print(f"Imbalance ratio: {counts.max() / counts.min():.2f}x  (max / min)")

Number of classes: 77

=== TRAIN support per class ===
shape: (77, 2)
┌───────┬───────┐
│ label ┆ count │
│ ---   ┆ ---   │
│ i64   ┆ u32   │
╞═══════╪═══════╡
│ 23    ┆ 35    │
│ 72    ┆ 41    │
│ 10    ┆ 59    │
│ 18    ┆ 61    │
│ 41    ┆ 82    │
│ …     ┆ …     │
│ 19    ┆ 177   │
│ 75    ┆ 180   │
│ 6     ┆ 181   │
│ 28    ┆ 182   │
│ 15    ┆ 187   │
└───────┴───────┘

=== TEST support per class ===
shape: (77, 2)
┌───────┬───────┐
│ label ┆ count │
│ ---   ┆ ---   │
│ i64   ┆ u32   │
╞═══════╪═══════╡
│ 38    ┆ 40    │
│ 9     ┆ 40    │
│ 73    ┆ 40    │
│ 68    ┆ 40    │
│ 67    ┆ 40    │
│ …     ┆ …     │
│ 44    ┆ 40    │
│ 32    ┆ 40    │
│ 76    ┆ 40    │
│ 1     ┆ 40    │
│ 36    ┆ 40    │
└───────┴───────┘

=== TRAIN imbalance summary ===
Total examples : 10003
Min support    : 35
Max support    : 187
Mean support   : 129.9
Imbalance ratio: 5.34x  (max / min)


### Multinomial Naive Bayes 
**For our baseline, let's use Naive Bayes with Bag of Words and TF-IDF.**

In [158]:
# 2. Pull out text + label (change these if your columns differ)
TEXT_COL, LABEL_COL = "text", "label"
train_texts, y_train = df_train[TEXT_COL].to_list(), df_train[LABEL_COL].to_list()
test_texts,  y_test  = df_test[TEXT_COL].to_list(),  df_test[LABEL_COL].to_list()

# 3. Bag of Words
bow = CountVectorizer()
X_train_bow = bow.fit_transform(train_texts)
X_test_bow  = bow.transform(test_texts)
y_pred_bow  = MultinomialNB().fit(X_train_bow, y_train).predict(X_test_bow)

# 4. TF-IDF
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(train_texts)
X_test_tfidf  = tfidf.transform(test_texts)
y_pred_tfidf  = MultinomialNB().fit(X_train_tfidf, y_train).predict(X_test_tfidf)

# 5. Metrics — macro (every class weighted equally, exposes imbalance)
for name, y_pred in [("Bag of Words", y_pred_bow), ("TF-IDF", y_pred_tfidf)]:
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_test, y_pred, average="macro", zero_division=0)
    print(f"{name:<15} Accuracy: {acc:.4f} | Macro-P: {prec:.4f} | Macro-R: {rec:.4f} | Macro-F1: {f1:.4f}")

Bag of Words    Accuracy: 0.7958 | Macro-P: 0.8251 | Macro-R: 0.7958 | Macro-F1: 0.7884
TF-IDF          Accuracy: 0.7886 | Macro-P: 0.8166 | Macro-R: 0.7886 | Macro-F1: 0.7731


##### Hyperparameter Tuning with GridSearchCV

We search for the best classifier configuration automatically instead of tuning by hand.

**Pipeline.** The vectorizer and `MultinomialNB` are chained together so the vectorizer is re-fitted *only* on the training portion of each fold — preventing data leakage into the vocabulary/IDF weights.

**Parameter grid.** A list of two branches (Bag of Words and TF-IDF) so we compare both representations and their settings at once.

*Shared vectorizer parameters:*
- `stop_words`: `["english", None]` — remove English stop words or keep everything
- `ngram_range`: `[(1,1), (1,2)]` — unigrams only, or unigrams + bigrams
- `min_df`: `[1, 2, 5]` — drop terms appearing in fewer than N documents
- `max_df`: `[1.0, 0.9]` — drop terms appearing in more than 90% of documents
- `max_features`: `[None, 5000, 10000]` — cap vocabulary to the top-N terms

*Branch-specific:*
- BoW only — `binary`: `[False, True]` — raw counts vs. presence/absence
- TF-IDF only — `sublinear_tf`: `[False, True]` — linear vs. log-scaled term frequency

*Model parameters (`MultinomialNB`):*
- `alpha`: `[0.1, 0.5, 1.0]` — Laplace/Lidstone smoothing strength
- `fit_prior`: `[True, False]` — learn skewed class priors from training data, or assume uniform (our imbalance lever)

**Search.** `GridSearchCV` tries every combination, scoring each with **5-fold cross-validation**. We rank by **macro-F1** so all 77 classes count equally, respecting the class imbalance.

**Evaluation.** The best config is retrained on the full training set and evaluated once on the held-out test set — untouched during search — giving an honest final score.

In [160]:
pipe = Pipeline([
    ("vec", CountVectorizer()),
    ("clf", MultinomialNB()),
])

# Vectorizer knobs shared by both branches
shared_vec = {
    "vec__stop_words":  ["english", None],
    "vec__ngram_range": [(1, 1), (1, 2)],
    "vec__min_df":      [1, 2, 5],          # drop ultra-rare terms (noise / overfit)
    "vec__max_df":      [1.0, 0.9],         # drop near-ubiquitous terms
    "vec__max_features":[None, 5000, 10000],# cap vocabulary size
}
shared_clf = {
    "clf__alpha":     [0.1, 0.5, 1.0],
    "clf__fit_prior": [True, False],
}

param_grid = [
    {"vec": [CountVectorizer()], "vec__binary": [False, True], **shared_vec, **shared_clf},
    {"vec": [TfidfVectorizer()], "vec__sublinear_tf": [False, True], **shared_vec, **shared_clf},
]

grid = GridSearchCV(pipe, param_grid, scoring="f1_macro", cv=5, n_jobs=-1, verbose=1)
grid.fit(train_texts, y_train)

print("\nBest CV macro-F1:", round(grid.best_score_, 4))
print("Best params:")
for k, v in grid.best_params_.items():
    print(f"   {k}: {v}")

y_pred = grid.predict(test_texts)
print("\n=== Test set (best model) ===")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

Fitting 5 folds for each of 1728 candidates, totalling 8640 fits


/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Best CV macro-F1: 0.8474
Best params:
   clf__alpha: 0.1
   clf__fit_prior: False
   vec: TfidfVectorizer()
   vec__max_df: 1.0
   vec__max_features: None
   vec__min_df: 1
   vec__ngram_range: (1, 2)
   vec__stop_words: None
   vec__sublinear_tf: True

=== Test set (best model) ===
              precision    recall  f1-score   support

           0     0.8605    0.9250    0.8916        40
           1     0.9756    1.0000    0.9877        40
           2     0.8333    1.0000    0.9091        40
           3     0.8571    0.9000    0.8780        40
           4     0.9737    0.9250    0.9487        40
           5     0.7632    0.7250    0.7436        40
           6     0.8372    0.9000    0.8675        40
           7     0.8222    0.9250    0.8706        40
           8     0.9286    0.9750    0.9512        40
           9     0.9091    1.0000    0.9524        40
          10     0.9286    0.6500    0.7647        40
          11     0.8718    0.8500    0.8608        40
          12

##### Parameters tuned in GridSearch

**Vectorizer (how text becomes numbers):**
- `sublinear_tf` (TF-IDF only) — instead of raw counts, applies `1 + log(tf)`, dampening heavily repeated words. Won `True`.
- `stop_words` — `"english"` removes common words ("the", "to", "my"); `None` keeps everything. Won `None`: in short queries these words carry meaning (`transfer to` vs `transfer from`).
- `ngram_range` — `(1,1)` single words only; `(1,2)` also word pairs. Won `(1,2)`.
- `min_df` — discards terms appearing in fewer than N documents (removes noise/typos). Won `1` (keeps everything).
- `max_df` — discards terms appearing in more than X% of documents (automatic stop-words). Won `1.0` (keeps everything).
- `max_features` — limits the vocabulary to the top-N most frequent terms. Won `None` (no limit).
- `binary` (BoW only) — real counts vs. presence/absence (0/1).

**Model (`MultinomialNB`):**
- `alpha` — smoothing that avoids zero probabilities (prevents a word never seen in a class from nullifying the prediction). `1.0` = smoother; `0.1` = trusts the data more. Won `0.1`.
- `fit_prior` — `True` learns the (imbalanced) class frequencies; `False` assumes all equally likely. This is the imbalance lever. Won `False`.

**Conclusion:** the dataset rewarded *keeping information* (no stop-word removal, no vocabulary trimming, bigrams on) and *not letting the model assume too much* (low smoothing, uniform prior).

**It would be interesting to check lower values for the alpha smoothing parameter. Lets fixe the other optimal parameters and test it.**

In [161]:
# Fixa os vencedores do vectorizer e varre só o alpha, agora com valores menores
pipe = Pipeline([
    ("vec", TfidfVectorizer(ngram_range=(1, 2), stop_words=None,
                            min_df=1, max_df=1.0, max_features=None,
                            sublinear_tf=True)),
    ("clf", MultinomialNB(fit_prior=False)),
])

param_grid = {"clf__alpha": [0.01, 0.03, 0.05, 0.1, 0.2, 0.3]}

grid = GridSearchCV(pipe, param_grid, scoring="f1_macro", cv=5, n_jobs=-1, verbose=1)
grid.fit(train_texts, y_train)

print("Melhor alpha:", grid.best_params_["clf__alpha"])
print("Macro-F1 (CV):", round(grid.best_score_, 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Melhor alpha: 0.03
Macro-F1 (CV): 0.8493


**0.03 end up being the best option! +0.003 on f1**

Let's Test the Robustness fro the optimal MNBeyes model.

In [ ]:
# ------------------------------------------------------------------
# Winning config
# ------------------------------------------------------------------
model = Pipeline([
    ("vec", TfidfVectorizer(ngram_range=(1, 2), stop_words=None,
                            min_df=1, max_df=1.0, max_features=None,
                            sublinear_tf=True)),
    ("clf", MultinomialNB(alpha=0.03, fit_prior=False)),
])
model.fit(train_texts, y_train)
y_pred = model.predict(test_texts)

# ==================================================================
# 1. FULL METRIC SUITE on the test set
# ==================================================================
print("=" * 55)
print("TEST-SET METRICS (winning config)")
print("=" * 55)
print(f"Accuracy                 : {accuracy_score(y_test, y_pred):.4f}\n")

for avg in ["macro", "micro", "weighted"]:
    p = precision_score(y_test, y_pred, average=avg, zero_division=0)
    r = recall_score(y_test, y_pred, average=avg, zero_division=0)
    f = f1_score(y_test, y_pred, average=avg, zero_division=0)
    print(f"{avg.capitalize():<9} | Precision: {p:.4f} | Recall: {r:.4f} | F1: {f:.4f}")

# ==================================================================
# 2. ROBUSTNESS A — Cross-validation stability on TRAIN
#    (does the score hold across different data splits?)
# ==================================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_macro_f1 = cross_val_score(model, train_texts, y_train,
                              scoring="f1_macro", cv=cv, n_jobs=-1)
print("\n" + "=" * 55)
print("ROBUSTNESS A — 5-fold CV macro-F1 on train")
print("=" * 55)
print(f"Per fold : {np.round(cv_macro_f1, 4)}")
print(f"Mean±Std : {cv_macro_f1.mean():.4f} ± {cv_macro_f1.std():.4f}")

# ==================================================================
# 3. ROBUSTNESS B — Bootstrap CI on the TEST set
#    (how much would the score wobble on resampled test data?)
# ==================================================================
rng = np.random.default_rng(42)
n = len(y_test)
boot_macro_f1, boot_acc = [], []
for _ in range(1000):
    idx = rng.integers(0, n, n)          # sample with replacement
    boot_macro_f1.append(f1_score(y_test[idx], y_pred[idx],
                                  average="macro", zero_division=0))
    boot_acc.append(accuracy_score(y_test[idx], y_pred[idx]))

def ci(vals):
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)

lo_f1, hi_f1   = ci(boot_macro_f1)
lo_acc, hi_acc = ci(boot_acc)
print("\n" + "=" * 55)
print("ROBUSTNESS B — Bootstrap 95% CI on test (1000 resamples)")
print("=" * 55)
print(f"Macro-F1 : {np.mean(boot_macro_f1):.4f}  95% CI [{lo_f1:.4f}, {hi_f1:.4f}]")
print(f"Accuracy : {np.mean(boot_acc):.4f}  95% CI [{lo_acc:.4f}, {hi_acc:.4f}]")

# ==================================================================
# 4. Per-class report (find the intents still struggling)
# ==================================================================
print("\n" + "=" * 55)
print("PER-CLASS REPORT")
print("=" * 55)
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

TEST-SET METRICS (winning config)
Accuracy                 : 0.8688

Macro     | Precision: 0.8731 | Recall: 0.8688 | F1: 0.8681
Micro     | Precision: 0.8688 | Recall: 0.8688 | F1: 0.8688
Weighted  | Precision: 0.8731 | Recall: 0.8688 | F1: 0.8681

ROBUSTNESS A — 5-fold CV macro-F1 on train
Per fold : [0.8538 0.8517 0.8601 0.8429 0.8519]
Mean±Std : 0.8521 ± 0.0055

ROBUSTNESS B — Bootstrap 95% CI on test (1000 resamples)
Macro-F1 : 0.8672  95% CI [0.8550, 0.8786]
Accuracy : 0.8692  95% CI [0.8578, 0.8805]

PER-CLASS REPORT
              precision    recall  f1-score   support

           0     0.9024    0.9250    0.9136        40
           1     0.9756    1.0000    0.9877        40
           2     0.8889    1.0000    0.9412        40
           3     0.8222    0.9250    0.8706        40
           4     1.0000    0.9250    0.9610        40
           5     0.7568    0.7000    0.7273        40
           6     0.8372    0.9000    0.8675        40
           7     0.8182    0.9000    

**Robustness A — stability (5-fold CV on train).**
Folds: `[0.8538, 0.8517, 0.8601, 0.8429, 0.8519]` → mean **0.8521 ± 0.0055**.
The tiny std means performance barely moves across data splits — the score is reliable, not a lucky split.

**Robustness B — precision (bootstrap 95% CI on test).**
Test macro-F1 = **0.8672**, 95% CI `[0.8550, 0.8786]` (~2.4 points wide).
Tight enough to honestly report **macro-F1 ≈ 0.87**.

*Note on the 95% CI:* it describes the reliability of the procedure — if the experiment were repeated many times, ~95% of the intervals built this way would contain the true macro-F1. It does **not** mean "95% chance the true value is in this specific interval" (the true value is fixed, not random).

**Two sanity checks that pass:**
- Accuracy (0.8692) ≈ macro-F1 (0.8672) — the balanced test set working as expected, and proof the imbalance handling worked (if rare classes were failing, macro-F1 would sag below accuracy; it doesn't).
- CV-on-train (0.8521) sits just *below* test (0.8672) — the healthy direction, no overfitting (overfitting would show train scores far *above* test).

### Support Vector Machines.

In [168]:
pipe = Pipeline([
    ("vec", TfidfVectorizer()),
    ("clf", LinearSVC(dual="auto", max_iter=5000)),
])

param_grid = [
    {   # TF-IDF branch
        "vec": [TfidfVectorizer()],
        "vec__ngram_range":  [(1, 1), (1, 2)],
        "vec__sublinear_tf": [True, False],
        "vec__min_df":       [1, 2],
        "clf__C":            [0.1, 0.5, 1.0, 5.0],
        "clf__class_weight": [None, "balanced"],   # native imbalance lever
    },
    {   # BoW branch
        "vec": [CountVectorizer()],
        "vec__ngram_range":  [(1, 1), (1, 2)],
        "vec__min_df":       [1, 2],
        "clf__C":            [0.1, 0.5, 1.0, 5.0],
        "clf__class_weight": [None, "balanced"],
    },
]

grid = GridSearchCV(pipe, param_grid, scoring="f1_macro", cv=5, n_jobs=-1, verbose=1)
grid.fit(train_texts, y_train)

print("\nBest CV macro-F1:", round(grid.best_score_, 4))
print("Best params:")
for k, v in grid.best_params_.items():
    print(f"   {k}: {v}")

y_pred = grid.predict(test_texts)
print("\n=== Test set (best model) ===")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

Fitting 5 folds for each of 96 candidates, totalling 480 fits

Best CV macro-F1: 0.8795
Best params:
   clf__C: 1.0
   clf__class_weight: None
   vec: TfidfVectorizer()
   vec__min_df: 1
   vec__ngram_range: (1, 2)
   vec__sublinear_tf: True

=== Test set (best model) ===
              precision    recall  f1-score   support

           0     0.9750    0.9750    0.9750        40
           1     0.9756    1.0000    0.9877        40
           2     1.0000    0.9750    0.9873        40
           3     0.9250    0.9250    0.9250        40
           4     1.0000    0.9750    0.9873        40
           5     0.6889    0.7750    0.7294        40
           6     0.8810    0.9250    0.9024        40
           7     0.9500    0.9500    0.9500        40
           8     0.9268    0.9500    0.9383        40
           9     0.9512    0.9750    0.9630        40
          10     0.9167    0.8250    0.8684        40
          11     0.8718    0.8500    0.8608        40
          12     0.8537 

**The SVM performs better achieving 0.8795 +0.03 than Beyes**

**Logisitc Regression**

In [ ]:
pipe = Pipeline([
    ("vec", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=5000)),
])

param_grid = [
    {   # TF-IDF branch
        "vec": [TfidfVectorizer()],
        "vec__ngram_range":  [(1, 1), (1, 2)],
        "vec__sublinear_tf": [True, False],
        "vec__min_df":       [1, 2],
        "clf__C":            [0.1, 0.5, 1.0, 5.0],
        "clf__class_weight": [None, "balanced"],   # native imbalance lever
    },
    {   # BoW branch
        "vec": [CountVectorizer()],
        "vec__ngram_range":  [(1, 1), (1, 2)],
        "vec__min_df":       [1, 2],
        "clf__C":            [0.1, 0.5, 1.0, 5.0],
        "clf__class_weight": [None, "balanced"],
    },
]

grid = GridSearchCV(pipe, param_grid, scoring="f1_macro", cv=5, n_jobs=-1, verbose=1)
grid.fit(train_texts, y_train)

print("\nBest CV macro-F1:", round(grid.best_score_, 4))
print("Best params:")
for k, v in grid.best_params_.items():
    print(f"   {k}: {v}")

y_pred = grid.predict(test_texts)
print("\n=== Test set (best model) ===")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

Fitting 5 folds for each of 96 candidates, totalling 480 fits

Best CV macro-F1: 0.8831
Best params:
   clf__C: 5.0
   clf__class_weight: balanced
   vec: TfidfVectorizer()
   vec__min_df: 1
   vec__ngram_range: (1, 1)
   vec__sublinear_tf: True

=== Test set (best model) ===
              precision    recall  f1-score   support

           0     1.0000    0.9500    0.9744        40
           1     0.9756    1.0000    0.9877        40
           2     0.9756    1.0000    0.9877        40
           3     0.9512    0.9750    0.9630        40
           4     1.0000    0.9250    0.9610        40
           5     0.6889    0.7750    0.7294        40
           6     0.9500    0.9500    0.9500        40
           7     0.9730    0.9000    0.9351        40
           8     0.9744    0.9500    0.9620        40
           9     0.9512    0.9750    0.9630        40
          10     0.8571    0.9000    0.8780        40
          11     0.8718    0.8500    0.8608        40
          12     0.8

**Logistic Regression end up being the best model: 0.8831**

Resumo:
- LR: 0.8831
- SVM: 0.8795
- NB: 0.8474

#  Dense Vectors

FastText with Pooling

In [171]:
import numpy as np
import gensim.downloader as api
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# ------------------------------------------------------------------
# 1. Load FastText vectors (first run downloads ~1 GB, then cached)
# ------------------------------------------------------------------
print("Loading FastText vectors...")
ft = api.load("fasttext-wiki-news-subwords-300")
DIM = 300

# ------------------------------------------------------------------
# 2. Encode: average the word vectors of each query into one dense vector
#    Minimal preprocessing (lowercase + split) — no stemming/lemmatisation,
#    embeddings already encode morphological relatedness.
#    Done ONCE (encoding is deterministic/unsupervised -> no leakage).
# ------------------------------------------------------------------
def embed(texts):
    out = np.zeros((len(texts), DIM), dtype=np.float32)
    for i, t in enumerate(texts):
        vecs = [ft[w] for w in t.lower().split() if w in ft]
        if vecs:
            out[i] = np.mean(vecs, axis=0)
    return out

print("Encoding train/test...")
X_train_ft = embed(train_texts)
X_test_ft  = embed(test_texts)
print("Dense shape:", X_train_ft.shape)   # (n_samples, 300)

# ------------------------------------------------------------------
# 3. Three models, each with its own grid, on the SAME dense features
# ------------------------------------------------------------------
configs = {
    "MultinomialNB": (
        # MinMaxScaler makes features non-negative so NB doesn't error
        Pipeline([("scale", MinMaxScaler()), ("clf", MultinomialNB())]),
        {"clf__alpha": [0.01, 0.1, 0.5, 1.0]},
    ),
    "LogisticRegression": (
        Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=5000))]),
        {"clf__C": [0.1, 0.5, 1.0, 5.0],
         "clf__class_weight": [None, "balanced"]},
    ),
    "LinearSVC": (
        Pipeline([("scale", StandardScaler()), ("clf", LinearSVC(dual="auto", max_iter=5000))]),
        {"clf__C": [0.1, 0.5, 1.0, 5.0],
         "clf__class_weight": [None, "balanced"]},
    ),
}

for name, (pipe, param_grid) in configs.items():
    print("\n" + "=" * 60)
    print(f"MODEL: {name}")
    print("=" * 60)

    grid = GridSearchCV(pipe, param_grid, scoring="f1_macro",
                        cv=5, n_jobs=-1, verbose=1)
    grid.fit(X_train_ft, y_train)

    print("Best CV macro-F1:", round(grid.best_score_, 4))
    print("Best params:")
    for k, v in grid.best_params_.items():
        print(f"   {k}: {v}")

    y_pred = grid.predict(X_test_ft)
    print("\n--- Test set ---")
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))

Loading FastText vectors...
[==================================================] 100.0% 958.5/958.4MB downloaded
Encoding train/test...
Dense shape: (10003, 300)

MODEL: MultinomialNB
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best CV macro-F1: 0.3315
Best params:
   clf__alpha: 0.01

--- Test set ---
              precision    recall  f1-score   support

           0     0.0882    0.2250    0.1268        40
           1     0.7714    0.6750    0.7200        40
           2     0.8462    0.5500    0.6667        40
           3     1.0000    0.1250    0.2222        40
           4     0.8571    0.3000    0.4444        40
           5     0.1905    0.3000    0.2330        40
           6     0.4545    0.5000    0.4762        40
           7     0.2963    0.2000    0.2388        40
           8     0.2597    0.5000    0.3419        40
           9     1.0000    0.3750    0.5455        40
          10     0.0000    0.0000    0.0000        40
          11     0.1042    0.12

/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best CV macro-F1: 0.7761
Best params:
   clf__C: 0.1
   clf__class_weight: None

--- Test set ---
              precision    recall  f1-score   support

           0     0.7333    0.8250    0.7765        40
           1     0.9500    0.9500    0.9500        40
           2     1.0000    0.9750    0.9873        40
           3     0.9412    0.8000    0.8649        40
           4     0.9394    0.7750    0.8493        40
           5     0.7045    0.7750    0.7381        40
           6     0.8571    0.9000    0.8780        40
           7     0.8056    0.7250    0.7632        40
           8     0.8605    0.9250    0.8916        40
           9     0.7872    0.9250    0.8506        40
          10     0.7500    0.6750    0.7105        40
          11     0.7111    0.8000    0.7529        40
          12     0.7692    0.7500    0.7595        40
          13     0.8810    0.9250    0.9024        40
          14     0.7368    0.7000    0.7179        40
          15     0.7778    0.8750    

#### Why the static dense results underperform TF-IDF

The FastText results came out **below** the TF-IDF baseline (~0.87). This is a known, expected outcome.

**The bottleneck is the pooling, not the classifier.** We tested three classifiers (NB, LogReg, LinearSVC) but they all consumed the *same* dense features, so the classifier was never the limiting factor. The information loss happens earlier, in how each query is turned into a single vector.

**Mean pooling blurs the signal.** FastText gives one vector *per word*, so to get one vector per query we average the word vectors (mean pooling). Averaging collapses the sentence into a "centre of mass" and, in doing so, discards word order and dilutes decisive words. In a query like *"lost my card"*, the informative word `card` is averaged together with filler like `my` and `lost`, so two intents that differ by a single key word end up at almost the same point.

**Banking77 is exactly the worst case for this.** The 77 intents are fine-grained and keyword-driven — they often differ by one or two decisive terms (`top up`, `PIN`, `IBAN`, `direct debit`). TF-IDF keeps each of those words as its own sharp feature, so it exploits precisely the signal that mean-pooled embeddings wash away. When the vocabulary *is* the signal, sparse lexical matching beats blurred semantic averaging.

**FastText also smears domain terms.** Its vectors were trained on Wikipedia/news, so precise banking terms get pulled toward their general meaning — losing the domain-specific sharpness the task depends on.

**MultinomialNB is additionally mismatched.** NB assumes count features; embeddings are continuous with negative values. The `MinMaxScaler` only makes it *run* — it doesn't make it appropriate — so its poor score is expected and not informative.

**What this does (and does not) prove.** The supported claim is narrow: *FastText with mean pooling underperforms TF-IDF on Banking77.* It does **not** prove "dense embeddings underperform TF-IDF" in general — we only tested one static model with one pooling method. Fairer tests still to run: TF-IDF-weighted pooling (lets decisive words dominate the average) and contextual embeddings (SBERT/MPNet), which avoid naive averaging by reading the sentence in context.

**BILSTM with FastText**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==================================================================
# 1. Build a vocabulary from the TRAINING tokens only
#    (index 0 = padding, index 1 = unknown word)
# ==================================================================
def tokenize(text):
    return text.lower().split()

PAD, UNK = 0, 1
vocab = {"<pad>": PAD, "<unk>": UNK}
for t in train_texts:
    for w in tokenize(t):
        if w not in vocab:
            vocab[w] = len(vocab)
print("Vocab size:", len(vocab))

def encode(text):
    return [vocab.get(w, UNK) for w in tokenize(text)]

# ==================================================================
# 2. Build the embedding matrix, seeding rows from FastText (`ft`)
#    Words FastText knows -> its 300-dim vector; unknown -> random.
# ==================================================================
EMB_DIM = 300
emb_matrix = np.random.normal(0, 0.1, (len(vocab), EMB_DIM)).astype(np.float32)
emb_matrix[PAD] = 0.0
hits = 0
for w, idx in vocab.items():
    if w in ft:                    # ft = the FastText KeyedVectors from before
        emb_matrix[idx] = ft[w]
        hits += 1
print(f"Seeded {hits}/{len(vocab)} words from FastText")
emb_matrix = torch.tensor(emb_matrix)

# ==================================================================
# 3. Dataset + padding collate function
# ==================================================================
class IntentDataset(Dataset):
    def __init__(self, texts, labels):
        self.data = [(torch.tensor(encode(t), dtype=torch.long), int(y))
                     for t, y in zip(texts, labels)]
    def __len__(self):  return len(self.data)
    def __getitem__(self, i):  return self.data[i]

def collate(batch):
    # drop any empty sequences, then pad to the longest in the batch
    batch = [(seq, y) for seq, y in batch if len(seq) > 0]
    seqs, ys = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded  = pad_sequence(seqs, batch_first=True, padding_value=PAD)
    return padded, lengths, torch.tensor(ys, dtype=torch.long)

# carve a validation set out of train to monitor training
tr_texts, va_texts, tr_y, va_y = train_test_split(
    train_texts, y_train, test_size=0.1, random_state=42, stratify=y_train)

train_loader = DataLoader(IntentDataset(tr_texts, tr_y), batch_size=64,
                          shuffle=True, collate_fn=collate)
val_loader   = DataLoader(IntentDataset(va_texts, va_y), batch_size=128,
                          shuffle=False, collate_fn=collate)
test_loader  = DataLoader(IntentDataset(test_texts, y_test), batch_size=128,
                          shuffle=False, collate_fn=collate)

# ==================================================================
# 4. The model:  Embedding -> BiLSTM -> last hidden -> Linear(77)
# ==================================================================
class BiLSTMClassifier(nn.Module):
    def __init__(self, emb_matrix, hidden=256, n_classes=77, dropout=0.4):
        super().__init__()
        vocab_size, emb_dim = emb_matrix.shape
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.embedding.weight.data.copy_(emb_matrix)   # seed with FastText
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True,
                            bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, n_classes)     # *2 for bidirectional

    def forward(self, x, lengths):
        emb = self.embedding(x)
        # pack so the LSTM ignores padding and gives the TRUE last state
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        # concat final forward (h_n[-2]) and backward (h_n[-1]) hidden states
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return self.fc(self.dropout(h))

model = BiLSTMClassifier(emb_matrix).to(device)

# class weights for the training imbalance (your imbalance lever)
counts = np.bincount(y_train, minlength=77)
weights = torch.tensor(len(y_train) / (77 * np.maximum(counts, 1)),
                       dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ==================================================================
# 5. Training loop with validation-based best-model saving
# ==================================================================
def run_epoch(loader, train=False):
    model.train() if train else model.eval()
    all_pred, all_true, total_loss = [], [], 0.0
    with torch.set_grad_enabled(train):
        for x, lengths, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x, lengths)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y)
            all_pred.extend(logits.argmax(1).cpu().numpy())
            all_true.extend(y.cpu().numpy())
    f1 = f1_score(all_true, all_pred, average="macro", zero_division=0)
    return total_loss / len(all_true), f1

EPOCHS = 15
best_val_f1, best_state = 0.0, None
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_f1 = run_epoch(train_loader, train=True)
    va_loss, va_f1 = run_epoch(val_loader, train=False)
    print(f"Epoch {ep:2d} | train loss {tr_loss:.3f} f1 {tr_f1:.3f} "
          f"| val loss {va_loss:.3f} f1 {va_f1:.3f}")
    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

# restore best model
model.load_state_dict(best_state)
print(f"\nBest val macro-F1: {best_val_f1:.4f}")

# ==================================================================
# 6. Final evaluation on the test set
# ==================================================================
model.eval()
y_pred, y_true = [], []
with torch.no_grad():
    for x, lengths, y in test_loader:
        logits = model(x.to(device), lengths)
        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_true.extend(y.numpy())

print("\n=== Test set (LSTM) ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}\n")
for avg in ["macro", "micro", "weighted"]:
    p = precision_score(y_true, y_pred, average=avg, zero_division=0)
    r = recall_score(y_true, y_pred, average=avg, zero_division=0)
    f = f1_score(y_true, y_pred, average=avg, zero_division=0)
    print(f"{avg.capitalize():<9} | Precision: {p:.4f} | Recall: {r:.4f} | F1: {f:.4f}")
print()
print(classification_report(y_true, y_pred, digits=4, zero_division=0))

Device: cpu
Vocab size: 4138
Seeded 2566/4138 words from FastText
Epoch  1 | train loss 3.712 f1 0.117 | val loss 2.535 f1 0.306
Epoch  2 | train loss 1.895 f1 0.463 | val loss 1.434 f1 0.583
Epoch  3 | train loss 1.064 f1 0.688 | val loss 1.029 f1 0.707
Epoch  4 | train loss 0.695 f1 0.802 | val loss 0.807 f1 0.761
Epoch  5 | train loss 0.482 f1 0.863 | val loss 0.715 f1 0.797
Epoch  6 | train loss 0.401 f1 0.888 | val loss 0.710 f1 0.795
Epoch  7 | train loss 0.266 f1 0.922 | val loss 0.646 f1 0.813
Epoch  8 | train loss 0.193 f1 0.946 | val loss 0.646 f1 0.810
Epoch  9 | train loss 0.164 f1 0.956 | val loss 0.705 f1 0.810
Epoch 10 | train loss 0.164 f1 0.955 | val loss 0.651 f1 0.828
Epoch 11 | train loss 0.139 f1 0.964 | val loss 0.643 f1 0.809
Epoch 12 | train loss 0.130 f1 0.966 | val loss 0.648 f1 0.828
Epoch 13 | train loss 0.074 f1 0.981 | val loss 0.636 f1 0.837
Epoch 14 | train loss 0.057 f1 0.989 | val loss 0.685 f1 0.824
Epoch 15 | train loss 0.054 f1 0.987 | val loss 0.64

**Best validation macro-F1:** 0.8378 → **Test:** Accuracy 0.8383 | Macro-F1 0.8381 | Weighted-F1 0.8381

The close match between validation (0.8378) and test (0.8381) is a good sign: the model generalises cleanly — no overfitting, and the validation-based model selection picked a config that held up on unseen data.

**The LSTM's advantage:** unlike the fixed mean-pooling of the FastText approach, the LSTM produces the document vector by *reading the words in order* (bidirectionally) and learning a task-specific summary. It replaces blind averaging with a **learned, order-aware compression** — preserving word order and tuning the representation to separate these 77 intents.

**Why the result still lands below TF-IDF (~0.87):** that advantage doesn't pay off here, and it's an expected outcome:
- **Too little data.** LSTMs are data-hungry; ~10k short queries isn't enough for the network to learn sequence patterns that beat a well-tuned sparse baseline.
- **Order matters little here.** Banking queries are short and keyword-driven — the decisive signal is *which* words appear, not their order. TF-IDF captures that directly; the LSTM's main strength (sequence modelling) has little to exploit.
- **Keyword precision lost.** Like all dense approaches, it trades exact term-matching for distributed semantics — a bad trade when the vocabulary itself is the signal.

**Note:** accuracy ≈ macro-F1 ≈ weighted-F1 (all ~0.838), confirming balanced per-class performance on the balanced test set — the imbalance handling (weighted loss) worked; no class is being silently ignored.

**Takeaway:** on this small, fine-grained, keyword-driven task, learned sequence compression (LSTM) still doesn't beat sparse lexical features (TF-IDF). The representation upgrade doesn't help when order carries little signal and data is limited.

# Context Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
# ------------------------------------------------------------------
# 2. Encode with SBERT (contextual embeddings)
#    Frozen pre-trained model -> encode once, no leakage, no manual pooling.
# ------------------------------------------------------------------
encoder = SentenceTransformer("all-mpnet-base-v2")   # runs on M1 MPS automatically

print("Encoding train...")
X_train = encoder.encode(train_texts, batch_size=64, show_progress_bar=True,
                         normalize_embeddings=True)
print("Encoding test...")
X_test  = encoder.encode(test_texts, batch_size=64, show_progress_bar=True,
                         normalize_embeddings=True)

print("Embedding shape:", X_train.shape)   # (n_samples, 768) — dense, contextual

# ------------------------------------------------------------------
# 3. Grid search on LogisticRegression (embeddings are fixed features)
# ------------------------------------------------------------------
clf = LogisticRegression(max_iter=5000)

param_grid = {
    "C":            [0.1, 0.5, 1.0, 5.0, 10.0],
    "class_weight": [None, "balanced"],
}

grid = GridSearchCV(clf, param_grid, scoring="f1_macro", cv=5, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("\nBest CV macro-F1:", round(grid.best_score_, 4))
print("Best params:")
for k, v in grid.best_params_.items():
    print(f"   {k}: {v}")

# ------------------------------------------------------------------
# 4. Evaluate on test set
# ------------------------------------------------------------------
y_pred = grid.predict(X_test)

print("\n=== Test set (SBERT + LogisticRegression) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
for avg in ["macro", "micro", "weighted"]:
    p = precision_score(y_test, y_pred, average=avg, zero_division=0)
    r = recall_score(y_test, y_pred, average=avg, zero_division=0)
    f = f1_score(y_test, y_pred, average=avg, zero_division=0)
    print(f"{avg.capitalize():<9} | Precision: {p:.4f} | Recall: {r:.4f} | F1: {f:.4f}")

print()
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14843.00it/s]


Encoding train...


Batches: 100%|██████████| 157/157 [00:42<00:00,  3.68it/s]


Encoding test...


Batches: 100%|██████████| 49/49 [00:11<00:00,  4.32it/s]

Embedding shape: (10003, 768)
Fitting 5 folds for each of 10 candidates, totalling 50 fits



Best CV macro-F1: 0.9271
Best params:
   C: 10.0
   class_weight: None

=== Test set (SBERT + LogisticRegression) ===
Accuracy: 0.9373

Macro     | Precision: 0.9395 | Recall: 0.9373 | F1: 0.9373
Micro     | Precision: 0.9373 | Recall: 0.9373 | F1: 0.9373
Weighted  | Precision: 0.9395 | Recall: 0.9373 | F1: 0.9373

              precision    recall  f1-score   support

           0     1.0000    0.9500    0.9744        40
           1     1.0000    1.0000    1.0000        40
           2     1.0000    1.0000    1.0000        40
           3     1.0000    1.0000    1.0000        40
           4     1.0000    0.9750    0.9873        40
           5     0.7391    0.8500    0.7907        40
           6     1.0000    0.9250    0.9610        40
           7     0.8750    0.8750    0.8750        40
           8     1.0000    1.0000    1.0000        40
           9     0.9756    1.0000    0.9877        40
          10     1.0000    0.9750    0.9873        40
          11     0.9189    0.8500

## SBERT + Logistic Regression Results

**Best CV macro-F1:** 0.9271 → **Test:** Accuracy 0.9373 | Macro-F1 0.9373 | Weighted-F1 0.9373

**This is the first approach to clearly beat TF-IDF (~0.87)** — a jump of ~6-7 points, and the best result so far.

**Why it wins where FastText and the LSTM failed:**
- **Contextual, not static.** SBERT reads each word *in context*, so "card" in "card payment" and "card arrived" get different vectors. FastText gave one fixed vector per word regardless of context.
- **No blind averaging.** SBERT pools token vectors internally in a *learned* way, trained specifically to produce good sentence representations — recovering the fine distinctions that FastText's mean-pooling washed out.
- **Semantic matching beats keyword matching here.** SBERT places queries with the same *meaning* close together even when they use different words — solving the synonym/paraphrase cases where TF-IDF (pure word-overlap) fails.

**Balanced performance confirmed:** accuracy ≈ macro-F1 ≈ weighted-F1 (all 0.9373) on the balanced test set — no class silently failing, so `class_weight=None` was sufficient (the embeddings separate classes well enough that the imbalance correction wasn't needed).

**Note:** best `C=10.0` sat at the top of the search range — the optimum may lie higher, so extending `C` to `[10, 20, 50]` could squeeze out a little more.

**Takeaway:** frozen contextual embeddings + a simple linear classifier outperform all classical sparse/static approaches. The representation — not the classifier — was the bottleneck all along.

In [8]:
# Install first (in your notebook or terminal):
# pip install transformers datasets accelerate

import numpy as np
import polars as pl
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

MODEL_NAME = "roberta-base"      # MPS-friendly
NUM_LABELS = 77

# Confirm MPS is available (should print True on your M1)
print("MPS available:", torch.backends.mps.is_available())

# ------------------------------------------------------------------
# 1. Load data + carve a validation split from TRAIN (test stays clean)
# ------------------------------------------------------------------
df_train = pl.read_parquet("/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/train-00000-of-00001.parquet")
df_test  = pl.read_parquet("/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/test-00000-of-00001.parquet")

TEXT_COL, LABEL_COL = "text", "label"
tr_texts, va_texts, tr_y, va_y = train_test_split(
    df_train[TEXT_COL].to_list(), df_train[LABEL_COL].to_list(),
    test_size=0.1, random_state=42, stratify=df_train[LABEL_COL].to_list())

train_ds = Dataset.from_dict({"text": tr_texts, "label": tr_y})
val_ds   = Dataset.from_dict({"text": va_texts, "label": va_y})
test_ds  = Dataset.from_dict({"text": df_test[TEXT_COL].to_list(),
                              "label": df_test[LABEL_COL].to_list()})

# ------------------------------------------------------------------
# 2. Tokenize
# ------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer)

# ------------------------------------------------------------------
# 3. Model
# ------------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS)

# ------------------------------------------------------------------
# 4. Metrics
# ------------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
    }

# ------------------------------------------------------------------
# 5. Training config — M1-friendly
# ------------------------------------------------------------------
args = TrainingArguments(
    output_dir="./roberta-banking77",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,              # validation during training (not test!)
    processing_class=tokenizer,       # renamed from tokenizer= in newer versions
    data_collator=collator,
    compute_metrics=compute_metrics,
)

# ------------------------------------------------------------------
# 6. Fine-tune, then evaluate ONCE on the held-out test set
# ------------------------------------------------------------------
trainer.train()

print("\n=== Validation (best model) ===")
print(trainer.evaluate(val_ds))

print("\n=== TEST (final, held-out) ===")
print(trainer.evaluate(test_ds))

MPS available: True


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12121.55it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752:

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall
1,1.484455,1.163247,0.836164,0.800454,0.839304,0.806630
2,0.700802,0.602931,0.894106,0.888129,0.903469,0.885402
3,0.483273,0.485519,0.910090,0.906241,0.915839,0.904258


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.79it/s]
/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.68it/s]
/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.67it/s]



=== Validation (best model) ===


...

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Macro Precision,Macro Recall
0.483273,0.485519,3,0.910090,0.906241,0.915839,0.904258


{'eval_loss': 0.4855193495750427, 'eval_accuracy': 0.9100899100899101, 'eval_macro_f1': 0.9062407225183754, 'eval_macro_precision': 0.9158388801924447, 'eval_macro_recall': 0.9042582823160604}

=== TEST (final, held-out) ===


/Users/gmonteiro/Dev/Customer-Support-Ticket-Classification/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Macro Precision,Macro Recall
0.483273,0.482481,3,0.918182,0.917943,0.921643,0.918182


{'eval_loss': 0.4824807047843933, 'eval_accuracy': 0.9181818181818182, 'eval_macro_f1': 0.917943345653309, 'eval_macro_precision': 0.9216429865669779, 'eval_macro_recall': 0.9181818181818179}
